In [26]:
import pandas as pd
import re

pd.set_option('display.max_rows', None);
pd.set_option('display.max_columns', None);

Load all the data

In [ ]:
#Links for reference

#Google Sheets 2018-2023
#2023
'https://docs.google.com/spreadsheets/d/1zlh79aLdicsDVjZoxfwVm8SY1iFlKm-GBI34TMk1I30/edit?usp=sharing'
#2022
'https://docs.google.com/spreadsheets/d/1mmsehBjl-V1eVueiCJEjFV6PhM6eKaCCPFIfcrk7IKk/edit?gid=0#gid=0'
#2021
'https://docs.google.com/spreadsheets/d/1E-Suwl9z_e8-W1JuP5HVmIKv6C4mpd73pFpgtCdrTkQ/edit?gid=0#gid=0'
#2020
'https://docs.google.com/spreadsheets/d/1Q5M7Zw_A-Kn2V7csyxGqleVXWliLVQ4e080aNI7oIiE/edit?gid=969721861#gid=969721861'
#2019
'https://docs.google.com/spreadsheets/d/1mY0ckZ7AKwlgeZv5uMA6eah7QwabyPv47SGNuqrO20I/edit?gid=969721861#gid=969721861'
#2018
'https://docs.google.com/spreadsheets/d/1nV6SU6eGIzi0_tz6ccezkEmtJewfioq3/edit?gid=1198915962#gid=1198915962'

#API endpoint 2016-2017
#2017
'https://data.colorado.gov/resource/uhi6-hddy.csv'
#2016
'https://data.colorado.gov/resource/m8vm-brgw.csv'

#Google Sheets 2007-2015
#2015
'https://docs.google.com/spreadsheets/d/1g4MnqPpjTFaYmhjIjeZUuOUdA3mkmvVntfcpD5gtRHY/edit?gid=1203697909#gid=1203697909'
#2014
'https://docs.google.com/spreadsheets/d/1Z6eI4edrGjrb2sJ_4gxYPljB7g60RkBk5BsMxyA_eH4/edit?gid=126874728#gid=126874728'
#2013
    #Intake
'https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1450511850#gid=1450511850
    #Outflow
'https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1492755857#gid=1492755857'
#2012
    #Intake
'https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=101086594#gid=101086594'
    #Outflow
'https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=805617810#gid=805617810'
#2011
    #Intake
'https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=1112172445#gid=1112172445'
    #Outflow
'https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=309603042#gid=309603042'
#2010
    #Intake
'https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1457683636#gid=1457683636'
    #Outflow
'https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1507785229#gid=1507785229'
#2009
    #Intake
'https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=831397415#gid=831397415'
    #Outflow
'https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=765780685#gid=765780685'
#2008
    #Intake
'https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=938588532#gid=938588532'
    #Outflow
'https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=1557235659#gid=1557235659'
#2007
'https://docs.google.com/spreadsheets/d/1CvEIJbREKwJRMFqkTOFWd2u6Bb_JZzhwLkU1JHdiYFE/edit?gid=994387581#gid=994387581'

#Licensing
'https://docs.google.com/spreadsheets/d/1q5PSJaDc-cpfKXBn6tQbg0jVxvwyrCNZUBk-5CCortA/edit?gid=531274251#gid=531274251'

In [3]:
# Helper function to convert Google Sheets URL to CSV export URL, skip converting API endpoints
def gsheet_to_csv_url(url):
    if url.endswith('.csv'):
        return url
    if "/edit" in url:
        base = url.split("/edit")[0]
        gid = "0"
        if "gid=" in url:
            gid = url.split("gid=")[-1].split("#")[0]
        return f"{base}/export?format=csv&gid={gid}"
    return url

# List of (year, url) tuples
sheet_links = [
    ("2023", "https://docs.google.com/spreadsheets/d/1zlh79aLdicsDVjZoxfwVm8SY1iFlKm-GBI34TMk1I30/edit?usp=sharing"),
    ("2022", "https://docs.google.com/spreadsheets/d/1mmsehBjl-V1eVueiCJEjFV6PhM6eKaCCPFIfcrk7IKk/edit?gid=0#gid=0"),
    ("2021", "https://docs.google.com/spreadsheets/d/1E-Suwl9z_e8-W1JuP5HVmIKv6C4mpd73pFpgtCdrTkQ/edit?gid=0#gid=0"),
    ("2020", "https://docs.google.com/spreadsheets/d/1Q5M7Zw_A-Kn2V7csyxGqleVXWliLVQ4e080aNI7oIiE/edit?gid=969721861#gid=969721861"),
    ("2019", "https://docs.google.com/spreadsheets/d/1mY0ckZ7AKwlgeZv5uMA6eah7QwabyPv47SGNuqrO20I/edit?gid=969721861#gid=969721861"),
    ("2018", "https://docs.google.com/spreadsheets/d/1nV6SU6eGIzi0_tz6ccezkEmtJewfioq3/edit?gid=1198915962#gid=1198915962"),
    ("2017", "https://data.colorado.gov/resource/uhi6-hddy.csv"), #api endpoint https://dev.socrata.com/foundry/data.colorado.gov/m8vm-brgw
    ("2016", "https://data.colorado.gov/resource/m8vm-brgw.csv"), #api endpoint https://dev.socrata.com/foundry/data.colorado.gov/uhi6-hddy
    ("2015", "https://docs.google.com/spreadsheets/d/1g4MnqPpjTFaYmhjIjeZUuOUdA3mkmvVntfcpD5gtRHY/edit?gid=1203697909#gid=1203697909"),
    ("2014", "https://docs.google.com/spreadsheets/d/1Z6eI4edrGjrb2sJ_4gxYPljB7g60RkBk5BsMxyA_eH4/edit?gid=126874728#gid=126874728"),
    ("2013 Intake", "https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1450511850#gid=1450511850"),
    ("2013 Outflow", "https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1492755857#gid=1492755857"),
    ("2012 Intake", "https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=101086594#gid=101086594"),
    ("2012 Outflow", "https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=805617810#gid=805617810"),
    ("2011 Intake", "https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=1112172445#gid=1112172445"),
    ("2011 Outflow", "https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=309603042#gid=309603042"),
    ("2010 Intake", "https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1457683636#gid=1457683636"),
    ("2010 Outflow", "https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1507785229#gid=1507785229"),
    ("2009 Intake", "https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=831397415#gid=831397415"),
    ("2009 Outflow", "https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=765780685#gid=765780685"),
    ("2008 Intake", "https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=938588532#gid=938588532"),
    ("2008 Outflow", "https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=1557235659#gid=1557235659"),
    ("2007", "https://docs.google.com/spreadsheets/d/1CvEIJbREKwJRMFqkTOFWd2u6Bb_JZzhwLkU1JHdiYFE/edit?gid=994387581#gid=994387581"),
    ("Licensing", "https://docs.google.com/spreadsheets/d/1q5PSJaDc-cpfKXBn6tQbg0jVxvwyrCNZUBk-5CCortA/edit?gid=531274251#gid=531274251"),
]

# Download and load each sheet as a DataFrame
dfs = {}
for year, url in sheet_links:
    csv_url = gsheet_to_csv_url(url)
    try:
        dfs[year] = pd.read_csv(csv_url)
        print(f"Loaded {year} data: {dfs[year].shape}")
    except Exception as e:
        print(f"Failed to load {year}: {e}")

# Example: display the first few rows of 2022 data
#dfs["2022"].head()

Loaded 2023 data: (347, 154)
Loaded 2022 data: (369, 154)
Loaded 2021 data: (354, 154)
Loaded 2020 data: (356, 154)
Loaded 2019 data: (349, 154)
Loaded 2018 data: (329, 173)
Loaded 2017 data: (279, 183)
Loaded 2016 data: (260, 204)
Loaded 2015 data: (257, 183)
Loaded 2014 data: (260, 213)
Loaded 2013 Intake data: (282, 47)
Loaded 2013 Outflow data: (282, 59)
Loaded 2012 Intake data: (282, 58)
Loaded 2012 Outflow data: (280, 57)
Loaded 2011 Intake data: (268, 58)
Loaded 2011 Outflow data: (269, 56)
Loaded 2010 Intake data: (264, 55)
Loaded 2010 Outflow data: (265, 57)
Loaded 2009 Intake data: (279, 66)
Loaded 2009 Outflow data: (280, 66)
Loaded 2008 Intake data: (276, 66)
Loaded 2008 Outflow data: (276, 67)
Loaded 2007 data: (297, 27)
Loaded Licensing data: (2974, 7)


Merge Intake and Outflow for years that have both

Get Facility Location Info

Parse geolocation from existing fields

In [20]:
# --- Split 'location_1' column in 2017 dataset into city, zip code, latitude, longitude ---

def parse_location1_2017(val):
    # Example: 'Denver, CO 80202\n(39.7392, -104.9903)'
    if pd.isnull(val):
        return pd.Series([None, None, None, None])
    parts = str(val).split('\n')
    #print(parts)
    #if len(parts) < 2:
    #    return pd.Series([None, None, None, None])
    city_zip = parts[1]
    #print(city_zip)
    # Second part: '(39.7392, -104.9903)'
    latlon = parts[2]
    # Extract city and zip
    city_zip_match = re.match(r'^(.*),\s*(\d{5})$', city_zip)
    if city_zip_match:
        city = city_zip_match.group(1)
        zip_code = city_zip_match.group(2)
    else:
        city = city_zip
        zip_code = None
    # Extract lat, lon
    latlon_match = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon)
    lat = float(latlon_match.group(1)) if latlon_match else None
    lon = float(latlon_match.group(2)) if latlon_match else None
    return pd.Series([city, zip_code, lat, lon])

#transform_2017 = dfs["2017"]

if "2017" in dfs and "location_1" in dfs["2017"].columns:
    dfs["2017"][["city_from_loc", "zip_from_loc", "lat", "lon"]] = dfs["2017"]["location_1"].apply(parse_location1_2017)

# Show the result
#dfs["2017"][["location_1", "city_from_loc", "zip_from_loc", "lat", "lon"]].head(20)

,location_1,city_from_loc,zip_from_loc,lat,lon
0,"\nVinita, OK, 74301\n(36.660071, -95.181344)","Vinita, OK",74301,36.660071,-95.181344
1,"\nARVADA, 80003\n(39.82682, -105.06527)",ARVADA,80003,39.826820,-105.065270
2,"\nArvada, 80007\n(39.839939, -105.186131)",Arvada,80007,39.839939,-105.186131
3,"\nArvada, 80007\n(39.839939, -105.186131)",Arvada,80007,39.839939,-105.186131
4,"\nAurora, 80011\n(39.74187, -104.799113)",Aurora,80011,39.741870,-104.799113
5,"\nAurora, 80011\n(39.74187, -104.799113)",Aurora,80011,39.741870,-104.799113
6,"\nAurora, 80011\n(39.74187, -104.799113)",Aurora,80011,39.741870,-104.799113
7,"\nAURORA, 80012\n(39.69879, -104.838565)",AURORA,80012,39.698790,-104.838565
8,"\nAurora, 80013\n(39.659198, -104.774814)",Aurora,80013,39.659198,-104.774814
9,"\nAurora, 80013\n(39.659198, -104.774814)",Aurora,80013,39.659198,-104.774814


In [ ]:
# --- Split 'location_1' column in 2016 dataset into city, latitude, longitude ---

def parse_location1(val):
    # Example: '\nLongmont, \n(40.165729, -105.101194)'
    if pd.isnull(val):
        return pd.Series([None, None, None])
    # Split on newline
    parts = str(val).split('\n')
    #print(parts)
    city = parts[1].strip(",    ")
    latlon = parts[2]
    # Extract lat, lon
    m2 = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon)
    lat = float(m2.group(1)) if m2 else None
    lon = float(m2.group(2)) if m2 else None
    return pd.Series([city, lat, lon])

#transform_2016 = dfs["2016"]

if "2016" in dfs and "location_1" in dfs["2016"].columns:
    dfs["2016"][["city_from_loc", "lat", "lon"]] = dfs["2016"]["location_1"].apply(parse_location1)

# Show the result
#dfs["2016"][["location_1", "city_from_loc", "lat", "lon"]].head(20)

,location_1,city_from_loc,lat,lon
0,"\nHartsel Colorado, \n(35.500801, -117.9478)",Hartsel Colorado,35.500801,-117.947800
1,"\nLongmont, \n(40.165729, -105.101194)",Longmont,40.165729,-105.101194
2,"\nLakewood, \n(39.710997, -105.088872)",Lakewood,39.710997,-105.088872
3,NaN,None,NaN,NaN
4,NaN,None,NaN,NaN
5,NaN,None,NaN,NaN
6,"\nLittleton, \n(39.612653, -105.016198)",Littleton,39.612653,-105.016198
7,"\nCedaredge, \n(38.900738, -107.923767)",Cedaredge,38.900738,-107.923767
8,"\nDelta, \n(38.741684, -108.070175)",Delta,38.741684,-108.070175
9,"\nFort Collins, \n(40.588972, -105.082459)",Fort Collins,40.588972,-105.082459


In [ ]:
# --- Parse 'Location 1' column in 2014 into street address, city, zip, latitude, longitude ---

def parse_location1_2014(val):
    # Example: '123 Main St\nDenver 80202\n(39.7392, -104.9903)'
    if pd.isnull(val):
        return pd.Series([None, None, None, None, None])
    parts = str(val).split('\n')
    #print(parts)
    if len(parts) < 2:
        return pd.Series([None, None, None, None, None])
    # First part: '123 Main St,'
    street = parts[0]
    # Second part: 'Denver 80202'
    city_zip = parts[1]
    # Third part: '(39.7392, -104.9903)'
    latlon_part = parts[2]
    # Extract city and zip
    city_zip_match = re.match(r'^(.*)\s*(\d{5})$', city_zip)
    if city_zip_match:
        city = city_zip_match.group(1)
        zip_code = city_zip_match.group(2)
    else:
        city = city_zip
        zip_code = None
    # Extract lat, lon
    latlon_match = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon_part)
    lat = float(latlon_match.group(1)) if latlon_match else None
    lon = float(latlon_match.group(2)) if latlon_match else None
    return pd.Series([street, city, zip_code, lat, lon])

#transform_2014 = dfs["2014"]

if "2014" in dfs and "Location 1" in dfs["2014"].columns:
    dfs["2014"][["street_address", "city_from_loc", "zip_from_loc", "lat", "lon"]] = dfs["2014"]["Location 1"].apply(parse_location1_2014)

# Show the result
#dfs["2014"][["Location 1", "street_address", "city_from_loc", "zip_from_loc", "lat", "lon"]].head(10)

,Location 1,street_address,city_from_loc,zip_from_loc,lat,lon
0,"3000 Jamaica Court, Suite 140\nAurora 80014\n(...","3000 Jamaica Court, Suite 140",Aurora,80014,39.661550,-104.862327
1,PO Box 60067\nColo Spgs 80960\n,PO Box 60067,Colo Spgs,80960,NaN,NaN
2,"2965 New Center Pt\nColorado Springs, Colorado...",2965 New Center Pt,"Colorado Springs, Colorado",80922,38.874109,-104.719462
3,9048 W 101st Ave\nWestminster 80021\n(39.88123...,9048 W 101st Ave,Westminster,80021,39.881233,-105.098262
4,1122 W. Myrtle St\nFt. Collins 80521\n(40.5805...,1122 W. Myrtle St,Ft. Collins,80521,40.580516,-105.096629
5,"10705 Fulton St\nBrighton 80601\n(39.891415, -...",10705 Fulton St,Brighton,80601,39.891415,-104.869934
6,PO Box 187\nDivide 80814\n,PO Box 187,Divide,80814,NaN,NaN
7,2613 Bradbury Court\nFort Collins 80521\n(40.5...,2613 Bradbury Court,Fort Collins,80521,40.569988,-105.125373
8,3121 Red Haven Way\nLittleton 80126\n(39.52130...,3121 Red Haven Way,Littleton,80126,39.521302,-104.951629
9,6010 W 88th Ave\nWestminster 80031\n(39.856434...,6010 W 88th Ave,Westminster,80031,39.856434,-105.063376


In [62]:
dfs["2007"].columns.tolist()

['Pacfa ID',
 'Name',
 'Address',
 'City',
 'State',
 'Zip',
 'Dogs accepted',
 'Dogs Returned',
 'Dogs Adopted',
 'Dogs Transferred/Facilities',
 'Dogs Euthanized',
 'Dogs Died',
 'Dogs Other',
 'Cats accepted',
 'Cats Returned',
 'Cats Adopted',
 'Cats Transferred/Facilities',
 'Cats Euthanized',
 'Cats Died',
 'Cats Other',
 'Other accepted',
 'Other Returned',
 'Other Adopted',
 'Other Transferred/Facilities',
 'Other Euthanized',
 'Other Died',
 'Others Other']

Combine location fields into single dataframe

In [ ]:
# Mapping of possible column names for each field
pacfa_id_cols = ['Pacfa ID',
                 'PACFA ID'
                 ]
license_no_cols = ['pacfa_license_number',
                   'PACFA License Number:'
                   ]
facility_cols = ['Facility Name', 
                 'Facility Name:',
                 'facility_name', 
                 'Name',
                 'NAME'
                 ]
address_cols = ['Facility Physical Street Address',
                'Facility Street address',
                'street_address',
                'Address',
                'ADDRESS'
                ]
city_cols = ['City', 
             'CITY',
             'city_from_loc'
             ]
state_cols = ['State', 
              'STATE'
              ]
zip_cols = ['zip_code',
            'Zip',
            'ZIP',
            'Zip Code',
            'zip_from_loc'
            ]
county_cols = ['County', 
               'county'
               ]
lat_cols = ['lat']
long_cols = ['lon']

def find_column(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

records = []
for year, df in dfs.items():
    cols = df.columns
    pcol = find_column(cols, pacfa_id_cols)
    lcol = find_column(cols, license_no_cols)
    fcol = find_column(cols, facility_cols)
    acol = find_column(cols, address_cols)
    ccol = find_column(cols, city_cols)
    scol = find_column(cols, state_cols)
    zcol = find_column(cols, zip_cols)
    kcol = find_column(cols, county_cols)
    latcol = find_column(cols, lat_cols)
    longcol = find_column(cols, long_cols)

    if fcol is None:
        continue
    sub = pd.DataFrame()
    sub['year'] = [year] * len(df)
    sub['pacfa_id'] = df[pcol] if pcol else None
    sub['facility_name'] = df[fcol]
    sub['address'] = df[acol] if acol else None
    sub['city'] = df[ccol] if ccol else None
    sub['state'] = df[scol] if scol else None
    sub['zip_code'] = df[zcol] if zcol else None
    sub['county'] = df[kcol] if kcol else None
    sub['latitude'] = df[latcol] if latcol else None
    sub['longitude'] = df[longcol] if longcol else None
    records.append(sub)

facilities_df = pd.concat(records, ignore_index=True)
facilities_df = facilities_df.drop_duplicates().dropna(subset=['facility_name'])
facilities_df.reset_index(drop=True, inplace=True)


/tmp/ipykernel_3368/628133913.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  facilities_df = pd.concat(records, ignore_index=True)


In [71]:
facilities_df.sort_values(by=['facility_name','year'])

,year,pacfa_id,facility_name,address,city,state,zip_code,county,latitude,longitude
3143,2014,None,104603,NaN,None,None,NaN,NaN,NaN,NaN
2631,2015,None,2 Blondes All Breed Rescue,7695 Louviers Blvd,None,None,80131,Douglas County,NaN,NaN
2390,2016,None,2 Blondes All Breed Rescue,None,Littleton,None,80126,Douglas County,39.612653,-105.016198
2153,2017,None,2 Blondes All Breed Rescue,None,Littleton,None,80126,Douglas County,39.543528,-104.960930
1420,2019,None,2 Blondes All Breed Rescue,None,None,None,None,None,NaN,NaN
1064,2020,None,"2 Blondes All Breed Rescue, Inc.",None,None,None,None,None,NaN,NaN
711,2021,None,"2 Blondes All Breed Rescue, Inc.",None,None,None,None,None,NaN,NaN
342,2022,None,"2 Blondes All Breed Rescue, Inc.",None,None,None,None,None,NaN,NaN
0,2023,None,"2 Blondes All Breed Rescue, Inc.",None,None,None,None,None,NaN,NaN
3142,2014,None,256,NaN,None,None,NaN,NaN,NaN,NaN
